# OaSC on Kaggle — OSDD reproduction notebook

**Paper:** *Recognizing Unseen States of Unknown Objects by Leveraging Knowledge Graphs* (WACV 2025)

This notebook is designed to get the **official OaSC method running on Kaggle** with as little dependency pain as possible.

### What this notebook reproduces

For OSDD, the authors' released setup uses:

- **visual backbone:** ResNet-101
- **semantic classifier weights:** generated from the knowledge graph / Tr-GCN stage
- **graph configuration:** `conceptnet_wordnet_hop1_thresh_10`
- **unseen states:** `empty`, `open`, `folded`, `filled`
- **seen states:** the remaining OSDD states
- **evaluation:** calibrated generalized zero-shot recognition, reporting Seen Accuracy, Unseen Accuracy, Harmonic Mean, and AUC

### Important Kaggle setting

Turn **Internet ON** and select a **GPU** before running. The notebook clones the authors' public repository and downloads their released material from the Google Drive ID used by their own `download_data.sh`.

> We intentionally do **not** install the repository's 2020-era full Conda environment (Python 3.7 / PyTorch 1.6). The pretrained OSDD evaluation only needs modern PyTorch/torchvision plus the authors' released embeddings and checkpoint, so this notebook uses a small compatibility layer instead.

In [ ]:
# Cell 1 — Kaggle / GPU sanity check
import os, sys, platform, subprocess
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())

try:
    import torch, torchvision
    print('PyTorch:', torch.__version__)
    print('Torchvision:', torchvision.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch import error:', repr(e))

if not torch.cuda.is_available():
    print('\nWARNING: GPU is not enabled. Kaggle -> Settings -> Accelerator -> GPU.')

In [ ]:
# Cell 2 — Install only the small helper dependency we need
%pip install -q gdown

## 1. Get the complete official repository

The ZIP you supplied contains the OaSC `KG/` implementation, README and environment file, but it does **not** contain several execution files referenced by the README (for example `test.py`, `finetune.py`, `flags.py`, and `download_data.sh`).

So on Kaggle we clone the **complete official repository** automatically.

In [ ]:
# Cell 3 — Clone the official repository
from pathlib import Path
import subprocess, shutil, os

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO_ROOT = WORK / 'OaSC_official'
METHOD_DIR = REPO_ROOT / 'OaSC'
SRC_DIR = METHOD_DIR / 'src'

if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/philipposg/OaSC.git',
        str(REPO_ROOT)
    ], check=True)
else:
    print('Repository already exists:', REPO_ROOT)

commit = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', REPO_ROOT)
print('Commit:', commit)
print('OaSC method dir:', METHOD_DIR)

required = [
    SRC_DIR / 'test.py',
    SRC_DIR / 'finetune.py',
    SRC_DIR / 'train_gnn.py',
    SRC_DIR / 'download_data.sh',
]
for p in required:
    print(('OK  ' if p.exists() else 'MISS'), p.relative_to(REPO_ROOT))

assert all(p.exists() for p in required), 'The full official repo was not cloned correctly.'

## 2. Download the authors' released OaSC material

The official `download_data.sh` points to Google Drive file ID:

`1i-jaMPKoBOmBncKBQ3qpsGQ9pLWgwRbz`

That archive contains the prepared datasets/materials, generated state embeddings, and released checkpoints. We use `gdown` because it is more reliable in notebooks than the old cookie-based `curl` script.

In [ ]:
# Cell 4 — Download OaSC_material.tar.gz (or reuse an uploaded copy)
import gdown, glob, shutil
from pathlib import Path

FILE_ID = '1i-jaMPKoBOmBncKBQ3qpsGQ9pLWgwRbz'
ARCHIVE = WORK / 'OaSC_material.tar.gz'

# Optional offline/manual fallback:
# If you already uploaded OaSC_material.tar.gz as a Kaggle Dataset,
# this searches /kaggle/input and uses it instead of downloading again.
manual_candidates = list(Path('/kaggle/input').rglob('OaSC_material.tar.gz')) if Path('/kaggle/input').exists() else []

if ARCHIVE.exists() and ARCHIVE.stat().st_size > 1_000_000:
    print('Using existing archive:', ARCHIVE)
elif manual_candidates:
    print('Using uploaded Kaggle archive:', manual_candidates[0])
    shutil.copy2(manual_candidates[0], ARCHIVE)
else:
    print('Downloading released OaSC material from the authors\' Google Drive...')
    out = gdown.download(id=FILE_ID, output=str(ARCHIVE), quiet=False)
    if out is None:
        raise RuntimeError(
            'Download failed. Make sure Kaggle Internet is ON, or upload '
            'OaSC_material.tar.gz as a Kaggle Dataset and rerun this cell.'
        )

print('Archive:', ARCHIVE)
print('Size: %.2f GB' % (ARCHIVE.stat().st_size / 1024**3))

In [ ]:
# Cell 5 — Extract the nested release archives into the locations expected by OaSC
import tarfile, os, shutil
from pathlib import Path

STAGING = WORK / 'oasc_release_staging'
STAGING.mkdir(parents=True, exist_ok=True)

marker = METHOD_DIR / '.oasc_material_extracted'

if marker.exists():
    print('Release material already extracted.')
else:
    print('Extracting outer archive...')
    with tarfile.open(ARCHIVE, 'r:gz') as tf:
        tf.extractall(STAGING)

    # Locate inner archives even if the outer archive adds one directory level.
    inner = {p.name: p for p in STAGING.rglob('*.tar.gz')}
    print('Inner archives found:', sorted(inner))

    def extract(name, destination):
        if name not in inner:
            print(f'WARNING: {name} not found; skipping.')
            return
        destination = Path(destination)
        destination.mkdir(parents=True, exist_ok=True)
        print(f'Extracting {name} -> {destination}')
        with tarfile.open(inner[name], 'r:gz') as tf:
            tf.extractall(destination)

    # Matches the intent of the authors' download_data.sh
    extract('datasets.tar.gz', METHOD_DIR)
    extract('embeddings.tar.gz', METHOD_DIR)
    extract('saved_checkpoints.tar.gz', METHOD_DIR)
    extract('data.tar.gz', SRC_DIR)

    marker.write_text('ok')

print('\nChecking expected OSDD files...')
paths = {
    'OSDD train': METHOD_DIR / 'datasets/osdd/train',
    'OSDD test': METHOD_DIR / 'datasets/osdd/test',
    'OaSC state embeddings': METHOD_DIR / 'embeddings/osdd_emb.pred',
    'Fine-tuned ResNet-101': METHOD_DIR / 'saved_checkpoints/osdd/finetuned_weights.pth',
}
for name, p in paths.items():
    print(f'{name:25s}:', 'OK' if p.exists() else 'MISSING', p)

missing = [name for name, p in paths.items() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing release material: ' + ', '.join(missing))

## 3. Inspect the released OaSC state classifier vectors

OaSC's KG/Tr-GCN branch predicts the final classifier parameters for each state. For ResNet-101 there are **2048 feature weights + 1 bias = 2049 numbers per state**.

The graph configuration used by the authors' OSDD test script is `conceptnet_wordnet_hop1_thresh_10`.

In [ ]:
# Cell 6 — Load and inspect the Tr-GCN-generated OSDD state vectors
import torch

EMB_PATH = METHOD_DIR / 'embeddings/osdd_emb.pred'
GRAPH_TYPE = 'conceptnet_wordnet_hop1_thresh_10'

# PyTorch >=2.6 changed torch.load's default behavior. These are trusted files
# downloaded from the paper authors' release, so we explicitly allow the old format.
def torch_load_compat(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:  # older PyTorch
        return torch.load(path, map_location=map_location)

emb_file = torch_load_compat(EMB_PATH)
print('Available graph keys:')
for k in emb_file.keys():
    print('  ', k)

assert GRAPH_TYPE in emb_file, f'{GRAPH_TYPE} not found in embedding file.'
state_vectors = emb_file[GRAPH_TYPE].float()
print('\nSelected graph:', GRAPH_TYPE)
print('State-vector tensor shape:', tuple(state_vectors.shape))
assert state_vectors.ndim == 2 and state_vectors.shape[1] == 2049

## 4. Load the OaSC visual model

We now reproduce the authors' testing logic in a Kaggle-safe form:

1. create a ResNet-101,
2. remove its classification head to obtain a 2048-D visual feature,
3. load the authors' fine-tuned visual backbone,
4. append a constant `1` to the feature,
5. multiply it by the KG-generated 2049-D state classifier vectors.

This is mathematically equivalent to a linear classifier whose weights and bias came from the KG/Tr-GCN stage.

In [ ]:
# Cell 7 — Build the OaSC inference model
import torch
import torch.nn as nn
from torchvision.models import resnet101

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_PATH = METHOD_DIR / 'saved_checkpoints/osdd/finetuned_weights.pth'

# Avoid downloading ImageNet weights here: the released fine-tuned checkpoint
# contains the trained backbone weights already.
try:
    cnn = resnet101(weights=None)
except TypeError:
    cnn = resnet101(pretrained=False)

cnn.fc = nn.Identity()
ckpt = torch_load_compat(CKPT_PATH)

# Some checkpoints may be wrapped; the released OaSC checkpoint is normally a state_dict.
if isinstance(ckpt, dict) and 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
    ckpt = ckpt['state_dict']

# Strip DataParallel prefix if present.
ckpt = {k.replace('module.', ''): v for k, v in ckpt.items()}
ckpt.pop('fc.weight', None)
ckpt.pop('fc.bias', None)

missing, unexpected = cnn.load_state_dict(ckpt, strict=False)
print('Missing keys:', missing)
print('Unexpected keys:', unexpected)

cnn = cnn.to(DEVICE).eval()
state_vectors = state_vectors.to(DEVICE)

print('Device:', DEVICE)
print('Visual feature dimension: 2048')
print('Classifier-vector shape:', tuple(state_vectors.shape))

## 5. Load OSDD and define the seen/unseen split

The authors' `test_osdd.sh` uses:

`empty_open_folded_filled`

as the omitted/unseen state set. The global OSDD state order used in their testing code is:

`closed, containing, empty, filled, folded, open, plugged, unfolded, unplugged`.

In [ ]:
# Cell 8 — Dataset + exact OSDD label mapping used by the authors
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

OSDD_CLASSES = [
    'closed', 'containing', 'empty', 'filled', 'folded',
    'open', 'plugged', 'unfolded', 'unplugged'
]
CLASS_TO_GLOBAL = {name: i for i, name in enumerate(OSDD_CLASSES)}

UNSEEN_NAMES = ['empty', 'open', 'folded', 'filled']
UNSEEN_IDS = [CLASS_TO_GLOBAL[x] for x in UNSEEN_NAMES]
SEEN_IDS = [i for i in range(len(OSDD_CLASSES)) if i not in UNSEEN_IDS]

transform = transforms.Compose([
    transforms.Resize([224, 224]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

TEST_DIR = METHOD_DIR / 'datasets/osdd/test'
test_ds = datasets.ImageFolder(TEST_DIR, transform=transform)

print('ImageFolder classes:', test_ds.classes)
print('Number of test images:', len(test_ds))

unknown_folders = [c for c in test_ds.classes if c not in CLASS_TO_GLOBAL]
assert not unknown_folders, f'Unexpected OSDD class folders: {unknown_folders}'

folder_idx_to_global = {
    folder_idx: CLASS_TO_GLOBAL[class_name]
    for class_name, folder_idx in test_ds.class_to_idx.items()
}

print('Seen states  :', [OSDD_CLASSES[i] for i in SEEN_IDS])
print('Unseen states:', [OSDD_CLASSES[i] for i in UNSEEN_IDS])
print('Folder -> global mapping:', folder_idx_to_global)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

## 6. Run OaSC inference once

The expensive CNN pass is done only once. After that, calibration over many values of \(\gamma\) is very fast.

The authors add \(\gamma\) to the scores of **unseen** classes and sweep it from `-5` to `15` in steps of `0.5`. This traces the seen–unseen trade-off used for HM and AUC.

In [ ]:
# Cell 9 — Extract scores for every OSDD test image
import numpy as np
from tqdm.auto import tqdm

all_scores = []
all_targets = []

with torch.inference_mode():
    for images, folder_labels in tqdm(test_loader, desc='OaSC inference'):
        images = images.to(DEVICE, non_blocking=True)
        features = cnn(images)  # [B, 2048]
        features_aug = torch.cat([
            features,
            torch.ones(features.shape[0], 1, device=features.device)
        ], dim=1)  # [B, 2049]

        # state_vectors: [9, 2049]; each row = [linear weights | bias]
        scores = features_aug @ state_vectors.T
        targets = torch.tensor(
            [folder_idx_to_global[int(x)] for x in folder_labels],
            dtype=torch.long
        )

        all_scores.append(scores.cpu())
        all_targets.append(targets)

scores = torch.cat(all_scores, dim=0)
targets = torch.cat(all_targets, dim=0)

print('Scores:', tuple(scores.shape))
print('Targets:', tuple(targets.shape))
assert scores.shape == (len(test_ds), len(OSDD_CLASSES))

In [ ]:
# Cell 10 — Exact seen/unseen calibration metrics used by the authors
import pandas as pd
import numpy as np

def evaluate_gamma(scores, targets, gamma):
    calibrated = scores.clone()
    calibrated[:, UNSEEN_IDS] += float(gamma)
    pred = calibrated.argmax(dim=1)

    seen_mask = torch.tensor([int(y) in SEEN_IDS for y in targets], dtype=torch.bool)
    unseen_mask = ~seen_mask

    seen_acc = (pred[seen_mask] == targets[seen_mask]).float().mean().item() if seen_mask.any() else float('nan')
    unseen_acc = (pred[unseen_mask] == targets[unseen_mask]).float().mean().item() if unseen_mask.any() else float('nan')
    hm = 0.0 if (seen_acc + unseen_acc) == 0 else 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc)
    total_acc = (pred == targets).float().mean().item()
    return total_acc, seen_acc, unseen_acc, hm

gammas = np.arange(-5.0, 15.125, 0.5)
rows = []
for gamma in gammas:
    total, seen, unseen, hm = evaluate_gamma(scores, targets, gamma)
    rows.append({
        'gamma': gamma,
        'total_acc': total,
        'seen_acc': seen,
        'unseen_acc': unseen,
        'harmonic_mean': hm,
    })

results = pd.DataFrame(rows)
best_idx = results['harmonic_mean'].idxmax()
best = results.loc[best_idx]

# Same orientation as the official test.py: np.trapz(seen_accuracy, unseen_accuracy)
try:
    auc = np.trapezoid(results['seen_acc'].to_numpy(), results['unseen_acc'].to_numpy())
except AttributeError:
    auc = np.trapz(results['seen_acc'].to_numpy(), results['unseen_acc'].to_numpy())

print('=== OaSC / OSDD reproduction ===')
print(f"Best gamma       : {best['gamma']:.2f}")
print(f"Seen @ best HM   : {100*best['seen_acc']:.2f}%")
print(f"Unseen @ best HM : {100*best['unseen_acc']:.2f}%")
print(f"Best HM          : {100*best['harmonic_mean']:.2f}%")
print(f"Max seen acc     : {100*results['seen_acc'].max():.2f}%")
print(f"Max unseen acc   : {100*results['unseen_acc'].max():.2f}%")
print(f"AUC              : {100*auc:.2f}")

summary = pd.DataFrame([{
    'Method': 'OaSC (released checkpoint)',
    'Graph': GRAPH_TYPE,
    'Unseen states': ', '.join(UNSEEN_NAMES),
    'Best gamma': best['gamma'],
    'Seen Acc (%)': 100*best['seen_acc'],
    'Unseen Acc (%)': 100*best['unseen_acc'],
    'HM (%)': 100*best['harmonic_mean'],
    'AUC': 100*auc,
}])
display(summary.round(3))

In [ ]:
# Cell 11 — Plot the seen–unseen trade-off
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.plot(results['unseen_acc'] * 100, results['seen_acc'] * 100, marker='o', markersize=3)
plt.scatter([best['unseen_acc'] * 100], [best['seen_acc'] * 100], s=80, label='Best HM')
plt.xlabel('Unseen accuracy (%)')
plt.ylabel('Seen accuracy (%)')
plt.title('OaSC on OSDD — seen/unseen calibration curve')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 7. Qualitative predictions

This section shows actual OSDD test images with the ground-truth and OaSC prediction at the best calibration value. These examples are useful for tomorrow's presentation because they show **what the model is recognizing**, not just one aggregate number.

In [ ]:
# Cell 12 — Visualize predictions at the best gamma
import random
import matplotlib.pyplot as plt

best_gamma = float(best['gamma'])
calibrated_scores = scores.clone()
calibrated_scores[:, UNSEEN_IDS] += best_gamma
preds = calibrated_scores.argmax(dim=1)

rng = random.Random(42)
N_SHOW = min(12, len(test_ds))
indices = rng.sample(range(len(test_ds)), N_SHOW)

# Denormalization for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.ravel()
for ax, idx in zip(axes, indices):
    image, folder_label = test_ds[idx]
    image_show = (image * std + mean).clamp(0, 1).permute(1,2,0).numpy()
    gt = int(targets[idx])
    pr = int(preds[idx])
    ok = gt == pr

    ax.imshow(image_show)
    ax.set_title(
        f"GT: {OSDD_CLASSES[gt]}\nPred: {OSDD_CLASSES[pr]} {'✓' if ok else '✗'}",
        fontsize=10
    )
    ax.axis('off')

plt.suptitle(f'OaSC qualitative predictions (gamma={best_gamma:.2f})', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Per-state accuracy and confusion matrix

This is not required by the authors' shell script, but it is very useful for understanding *which* states are difficult and for deciding what improvement to try next.

In [ ]:
# Cell 13 — Per-state accuracy + confusion matrix
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cm = confusion_matrix(targets.numpy(), preds.numpy(), labels=list(range(len(OSDD_CLASSES))))
per_state = []
for i, name in enumerate(OSDD_CLASSES):
    total_i = cm[i].sum()
    acc_i = (cm[i, i] / total_i) if total_i else np.nan
    per_state.append({
        'state': name,
        'split': 'unseen' if i in UNSEEN_IDS else 'seen',
        'samples': int(total_i),
        'accuracy_%': 100 * acc_i,
    })

per_state_df = pd.DataFrame(per_state)
display(per_state_df.round(2))

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm)
ax.set_xticks(range(len(OSDD_CLASSES)), OSDD_CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(OSDD_CLASSES)), OSDD_CLASSES)
ax.set_xlabel('Predicted state')
ax.set_ylabel('True state')
ax.set_title('OaSC OSDD confusion matrix')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# ✅ At this point, OaSC is working

You now have a complete **released-model reproduction path**:

`OSDD image → fine-tuned ResNet-101 feature → KG/Tr-GCN-generated state weights → state score → seen/unseen calibration`

For tomorrow's presentation, save the output of:

1. the summary metric table,
2. the seen/unseen trade-off curve,
3. the qualitative prediction grid,
4. the per-state results/confusion matrix.

---

# Optional: fine-tune the visual backbone yourself

The released embeddings already represent the KG/Tr-GCN output. The code below lets you train the **visual ResNet stage** on the OSDD training folders while freezing the KG-generated classifier head, matching the intent of the original OaSC fine-tuning procedure.

For a presentation tomorrow, keep `RUN_FINETUNE = False` until the pretrained reproduction above works. Then turn it on only if you have time.

In [ ]:
# Cell 14 — OPTIONAL: fine-tune ResNet-101 using the released Tr-GCN state weights
RUN_FINETUNE = False
FINETUNE_EPOCHS = 10       # paper script uses 150; start small for debugging
FINETUNE_BATCH_SIZE = 32
LR = 1e-4

if RUN_FINETUNE:
    from torchvision.models import resnet101, ResNet101_Weights
    from torch.utils.data import DataLoader, Dataset
    import torch.nn as nn
    import torch.optim as optim
    from tqdm.auto import tqdm

    TRAIN_DIR = METHOD_DIR / 'datasets/osdd/train'
    base_train_ds = datasets.ImageFolder(TRAIN_DIR, transform=transform)
    print('Folders present in training directory:', base_train_ds.classes)

    # Enforce the zero-shot protocol even if the downloaded train directory
    # happens to contain folders for all nine states: only SEEN states are used.
    seen_names_in_train = [name for name in base_train_ds.classes if name in [OSDD_CLASSES[i] for i in SEEN_IDS]]
    local_seen_name_to_label = {name: i for i, name in enumerate(seen_names_in_train)}

    kept = []
    for sample_idx, (_, folder_label) in enumerate(base_train_ds.samples):
        name = base_train_ds.classes[folder_label]
        if name in local_seen_name_to_label:
            kept.append((sample_idx, local_seen_name_to_label[name]))

    class SeenOnlyDataset(Dataset):
        def __init__(self, base, kept_pairs):
            self.base = base
            self.kept_pairs = kept_pairs
            self.classes = seen_names_in_train
        def __len__(self):
            return len(self.kept_pairs)
        def __getitem__(self, idx):
            base_idx, new_label = self.kept_pairs[idx]
            image, _ = self.base[base_idx]
            return image, new_label

    train_ds = SeenOnlyDataset(base_train_ds, kept)
    print('Seen states used for fine-tuning:', train_ds.classes)
    print('Training images after seen-only filtering:', len(train_ds))

    # Select the OaSC classifier vector corresponding to each local training class.
    train_global_ids = [CLASS_TO_GLOBAL[name] for name in train_ds.classes]
    fc_vectors = state_vectors.detach()[train_global_ids].clone()

    try:
        model = resnet101(weights=ResNet101_Weights.IMAGENET1K_V1)
    except Exception:
        model = resnet101(pretrained=True)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, len(train_ds.classes))
    with torch.no_grad():
        model.fc.weight.copy_(fc_vectors[:, :-1].cpu())
        model.fc.bias.copy_(fc_vectors[:, -1].cpu())

    # OaSC: keep KG-predicted final classifier parameters fixed while adapting
    # the visual feature extractor using images of seen states only.
    model.fc.weight.requires_grad = False
    model.fc.bias.requires_grad = False
    model = model.to(DEVICE)

    loader = DataLoader(
        train_ds,
        batch_size=FINETUNE_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    optimizer = optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        momentum=0.9,
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(FINETUNE_EPOCHS):
        model.train()
        running_loss, correct, n = 0.0, 0, 0
        bar = tqdm(loader, desc=f'Epoch {epoch+1}/{FINETUNE_EPOCHS}')
        for images, labels in bar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            n += images.size(0)
            bar.set_postfix(loss=running_loss/n, acc=correct/n)

    out_path = WORK / 'oasc_osdd_refinetuned_resnet101.pth'
    torch.save(model.state_dict(), out_path)
    print('Saved:', out_path)
else:
    print('Skipping fine-tuning. Set RUN_FINETUNE=True when you are ready.')

## Next improvement hook — do not change this until baseline results are saved

Once the reproduction is stable, the clean experimental sequence is:

1. **Baseline:** original released OaSC results.
2. **Modification:** change only the KG/state-vector generation or graph construction.
3. Recompute the state vectors.
4. Keep the same OSDD split, visual training protocol, and evaluation code.
5. Compare Seen / Unseen / HM / AUC against the baseline.

That gives you a controlled ablation rather than changing several components at once.

For the semantic graph-pruning idea, the next code target is the graph construction / concept selection stage under `OaSC/src/KG/` and the GNN pipeline in `train_gnn.py`; **do not modify the evaluation cells above**. That way your result comparison remains fair.